# LLM Preference Elicitation with GRUMs — Colors Domain

## 1. Introduction & Context

### What is this notebook about?

This notebook presents the results of applying **Generalized Rank-Utility Models (GRUMs)** to preference elicitation from Large Language Models (LLMs). The domain used here is a set of five colors — **Blue, Red, Green, Purple, Yellow** — chosen as a simple, controlled testbed to validate the methodology before scaling to richer domains (e.g., laptops).

### How does this differ from the reproduction notebook?

The companion notebook `1_reproduction.ipynb` **reproduced** the original GRUM paper's figures. In that setting, agents are real or synthetically sampled personas with features drawn from the dataset itself.

Here, we make a fundamentally different modeling choice:

| Aspect | Reproduction (`1_reproduction.ipynb`) | **This notebook** |
|---|---|---|
| **Agent source** | Personas from the dataset | LLM prompt contexts |
| **Agent features** | Dataset-supplied feature vectors (demographics for Sushi; latent coords for synthetic ds0–2) | PCA of LLM internal representations: Hidden State (`HS`) or Sentence Transformer PCA (`ST`) |
| **Preference data** | Synthetic (ds0–2) or real human rankings (Sushi) | LLM pairwise choices (via perplexity scoring) |
| **Goal** | Validate GRUM implementation against paper | Validate GRUM for LLM persona elicitation |

### Experiment Dimensions

We test across three independent axes:

| Dimension | Values |
|---|---|
| **LLM Model** | `Qwen2.5-0.5B` (Pretrained) · `Qwen2.5-0.5B-Instruct` (Instruct) |
| **Agent Embedding** | Hidden State PCA (`HS`) · Sentence Transformer PCA (`ST`) |
| **Elicitation Criterion** | `social` · `personalized` · `random` |

This yields **12 experimental configurations** in total.

### Three Conjectures

1. **Intrinsic alignment**: GRUM's $\delta$ will correlate strongly with BT's $\beta$ — BT captures the *average* preference.
2. **Persona effect**: The interaction matrix $B$ will be non-zero, producing *rank reversals* under different prompt contexts.
3. **Predictive advantage**: GRUM will achieve lower NLL than BT on held-out prompt contexts.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics.pairwise import cosine_similarity
from itertools import combinations

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

# ── Domain constants ──────────────────────────────────────────────────────────
COLOR_NAMES   = ["blue", "red", "green", "purple", "yellow"]
COLOR_PALETTE = {
    "blue": "#1f77b4", "red": "#d62728", "green": "#2ca02c",
    "purple": "#9467bd", "yellow": "#d6c427"
}
CRITERIA      = ["Social", "Personalized", "Random"]
EMBEDDINGS    = ["HS", "ST"]
MODELS        = ["Pretrained", "Instruct"]

print("Setup complete.")

## 2. Data Loading

We load all 12 result JSON files from the main experiment run. Each file captures:
- **Metadata**: model, embedding method, elicitation criterion, PCA dimension
- **`history`**: per-training-step snapshots of GRUM parameters ($\delta$, $B$) and BT weights ($\beta$)
- **`timing`**: wall-clock training duration

From each run we extract:
1. The **final-step** GRUM $\delta$ (intrinsic preferences) and $B$ (interaction matrix)
2. The **final-step** BT $\beta$ weights
3. The full **history** (for convergence plots)

In [ ]:
RESULTS_DIR = Path("../results/llm/llm_colors-20260326-114531/outputs")

def embed_label(method: str) -> str:
    if "hidden_state" in method:
        return "HS"
    if "sentence_transformer" in method:
        return "ST"
    return method

def model_label(model_id: str) -> str:
    return "Instruct" if "Instruct" in model_id else "Pretrained"

raw_results = []

for fpath in sorted(RESULTS_DIR.glob("*.json")):
    with open(fpath) as f:
        data = json.load(f)

    history = data["history"]
    last    = history[max(history.keys(), key=int)]

    delta_grum = np.array(last["grum"]["delta"])
    B          = np.array(last["grum"]["interaction"])
    beta_bt    = np.array(last["bt"]["delta"])   # BT weights (beta)

    # Full history for convergence plots
    steps_sorted    = sorted(history.keys(), key=int)
    delta_hist_grum = np.array([history[s]["grum"]["delta"] for s in steps_sorted])
    beta_hist_bt    = np.array([history[s]["bt"]["delta"]   for s in steps_sorted])

    raw_results.append({
        "Model":     model_label(data["model_id"]),
        "Embedding": embed_label(data["embedding_method"]),
        "Criterion": data["criterion"].capitalize(),
        "model_id":  data["model_id"],
        "pca_dim":   data["pca_dim"],
        "steps":     data["steps"],
        "delta_grum":      delta_grum,
        "B":               B,
        "beta_bt":         beta_bt,
        "delta_hist_grum": delta_hist_grum,
        "beta_hist_bt":    beta_hist_bt,
        "timing":    data.get("timing", {}),
    })

print(f"Loaded {len(raw_results)} runs.")
df_meta = pd.DataFrame([{
    k: v for k, v in r.items()
    if k not in {"delta_grum", "B", "beta_bt", "delta_hist_grum", "beta_hist_bt"}
} for r in raw_results])
df_meta[["Model", "Embedding", "Criterion", "pca_dim", "steps"]]

---

## 3. Experiment 1 — Intrinsic Preferences: GRUM δ vs BT β

**Conjecture**: The GRUM's intrinsic parameters $\delta$ will strongly correlate with the BT weights $\beta$.

In [ ]:
# ── Figure 1: GRUM δ vs BT β scatter ─────────────
fig, axes = plt.subplots(3, 4, figsize=(20, 13))
axes = axes.flatten()

corr_records = []

for r in raw_results:
    pcc, _ = pearsonr(r["delta_grum"], r["beta_bt"])
    corr_records.append({
        "Model":      r["Model"],
        "Embedding":  r["Embedding"],
        "Criterion":  r["Criterion"],
        "Pearson r":  pcc,
    })

df_corr_summary = pd.DataFrame(corr_records)
display(df_corr_summary)
print("Conjecture 1 correlation summary above.")

---

## 4. Embedding & Model Comparisons

Does the choice of internal representation (agent embedding) change what the model "thinks" the average preference is? And how does the interaction strength—the degree to which prompt context affects choice—vary between base and instruct-tuned models?

In [ ]:
# ── Aggregate delta by Model & Embedding (avg across criteria) ───────────────
delta_records = []
for r in raw_results:
    for i, name in enumerate(COLOR_NAMES):
        delta_records.append({
            "Model": r["Model"],
            "Embedding": r["Embedding"],
            "Color": name.capitalize(),
            "Delta": r["delta_grum"][i]
        })

df_delta = pd.DataFrame(delta_records)
df_delta_avg = df_delta.groupby(["Model", "Embedding", "Color"]).mean().reset_index()
display(df_delta_avg.head(10))
print("Intrinsic preference summary (partial) above.")

In [ ]:
# ── Figure 4: Interaction Strength (Frobenius Norm of B) ──────────────────────
b_norms = []
for r in raw_results:
    b_norms.append({
        "Model":     r["Model"],
        "Embedding": r["Embedding"],
        "Criterion": r["Criterion"],
        "B_norm":    np.linalg.norm(r["B"])
    })

df_b_summary = pd.DataFrame(b_norms).groupby(["Model", "Embedding"])["B_norm"].mean().reset_index()
display(df_b_summary)
print("Interaction strength (B norm) summary above.")

---

## 5. Elicitation Criterion Comparison

Does the choice of which agents and alternatives to query first influence the final preferences elicited from the model? We compare the **Social Choice**, **Personalized Choice**, and **Random** criteria.

In [ ]:
# ── Final delta norm comparison (Separated by Model) ──────────────────────────
df_final_d = pd.DataFrame([{
    "Model":     r["Model"],
    "Embedding": r["Embedding"],
    "Criterion": r["Criterion"],
    "delta_norm": np.linalg.norm(r["delta_grum"])
} for r in raw_results])

g = sns.catplot(
    data=df_final_d, x="Criterion", y="delta_norm", 
    hue="Embedding", col="Model", kind="bar",
    height=4.5, aspect=1.1, palette="muted"
)
g.set_axis_labels("Criterion", "Intrinsic Preference Strength (||δ||)")
plt.show()

display(df_final_d.groupby(["Model", "Embedding", "Criterion"])["delta_norm"].mean().unstack())
print("Intrinsic preference strength summary above.")

PLACEHOLDER: Waiting for data summary to write analysis.

In [ ]:
# ── Convergence of delta norm (Select: Pretrained | HS) ───────────────────────
select_configs = [r for r in raw_results if r["Model"] == "Pretrained" and r["Embedding"] == "HS"]

conv_data = []
for r in select_configs:
    norms = [np.linalg.norm(d) for d in r["delta_hist_grum"]]
    conv_data.append({"Criterion": r["Criterion"], "Final Norm": norms[-1], "Start Norm": norms[0]})

df_conv_summary = pd.DataFrame(conv_data)
display(df_conv_summary)
print("Convergence summary (Pretrained | HS) above.")

PLACEHOLDER: Waiting for data summary to write analysis.